# Loading and Querying a Very Large CSV Without Crashing

This notebook walks through a common real-world problem: you're handed a CSV
that's too big to comfortably open in Excel (or even in plain `pandas`), and
you need to explore it, filter it, and pull out useful subsets.

**Dataset:** [The Metropolitan Museum of Art's Open Access CSV](https://github.com/metmuseum/openaccess)
(`MetObjects.csv`) — a catalog of ~470,000+ museum objects, roughly **300+ MB**
as a single CSV. This is exactly the kind of file that makes Excel choke and
can make a careless `pandas.read_csv()` call eat all your RAM.

**What we'll cover:**
1. Getting the file onto disk
2. Why naively loading it can be a problem
3. Previewing it safely with **DuckDB** (SQL on top of the raw file, no full load)
4. Filtering and aggregating with SQL
5. Pulling a small, safe subset into `pandas` for further analysis
6. A quick look at the **Polars** alternative

> 💡 **Core idea:** DuckDB and Polars can query a CSV file *lazily*, scanning
> only what's needed from disk, instead of forcing the whole file into memory
> the way `pandas.read_csv()` does by default.


## Step 0: Install what we need

We'll use `duckdb` for SQL-on-CSV querying and `polars` for the lazy-dataframe
comparison at the end. Both are lightweight, pure pip installs — no database
server, no extra setup.


In [15]:
# Run this once. If you're in a shared classroom environment, this may already
# be installed.
%pip install -q duckdb polars requests


Note: you may need to restart the kernel to use updated packages.


## Step 1: Download the dataset

`MetObjects.csv` is stored on GitHub using **Git LFS**, which means the
regular "raw" GitHub URL won't give you the actual file — it'll give you a
tiny pointer file instead. We need the LFS media URL.

We'll stream the download to disk in chunks, rather than pulling the whole
thing into memory first, since the file is ~300 MB.


In [17]:
import os
import requests

DATA_URL = "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv"
DATA_PATH = "MetObjects.csv"

if not os.path.exists(DATA_PATH):
    print("Downloading MetObjects.csv (this is a few hundred MB, may take a minute)...")
    with requests.get(DATA_URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(DATA_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r{downloaded / total:.0%}", end="")
    print("\nDone.")
else:
    print("File already downloaded.")

size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")


100%
Done.
File size: 302.9 MB


## Step 2: Why not just `pandas.read_csv()`?

This is the "Excel-style crash" moment, just in Python instead. Loading the
*entire* file into a `pandas` DataFrame pulls every row and column into RAM at
once. On a few hundred MB this might actually work on a modern laptop — but
it's slow, it doesn't scale, and on a truly large file (multi-GB) or a
memory-constrained machine, it's exactly what kills the kernel.

**We're intentionally *not* running the naive version below on the full file.**
Feel free to try it yourself and watch your memory usage climb, but the rest
of this notebook shows the better way.


In [18]:
# DON'T casually run this on very large files — shown here only for comparison.
# import pandas as pd
# df = pd.read_csv(DATA_PATH)   # loads ALL ~470k rows x 50+ columns into memory at once


## Step 3: Preview safely with DuckDB

[DuckDB](https://duckdb.org/) is an embedded analytical database. It can run
SQL directly against a CSV file **on disk**, scanning only the rows/columns it
actually needs for a given query — it never loads the full file into memory
unless you explicitly ask it to.

No server, no config: just `import duckdb` and point it at the file path.


In [19]:
import duckdb

# Peek at the first few rows without loading the whole file
duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
    LIMIT 5
""").show()


┌───────────────┬──────────────┬──────────────────┬──────────────────┬───────────┬────────────────┬───────────────────┬───────────────┬─────────────┬──────────────────────────────┬─────────┬─────────┬─────────┬─────────┬───────────┬────────────────┬─────────────┬───────────────┬───────────────────────┬──────────────────────────────────────────────────────────────────────────────┬───────────────┬────────────────────────┬────────────────────┬───────────────────┬─────────────────┬───────────────┬────────────────────────────────────────────┬────────────────────────────────────────┬─────────────┬───────────────────┬─────────────────┬─────────┬──────────────────────────┬────────────────────────────────────┬────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬─────────┬─────────┬────────────┬─────────┬────────────────┬─────────────────────────┬──────────────────────────────────────────────────┬─────────────────────┬───────────────┬────────────────────────────────────

In [20]:
# Check the shape of the dataset (row count, column count) --
# DuckDB scans the file efficiently to compute this.
row_count = duckdb.sql(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_csv_auto('{DATA_PATH}')
""").fetchone()[0]

col_count = len(duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
    LIMIT 0
""").columns)

print(f"Rows: {row_count:,}")
print(f"Columns: {col_count}")


Rows: 484,956
Columns: 54


In [21]:
# See the column names and inferred types
duckdb.sql(f"""
    DESCRIBE SELECT * FROM read_csv_auto('{DATA_PATH}')
""").show(max_rows=60)


┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Object Number           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Is Highlight            │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Is Timeline Work        │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Is Public Domain        │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ Object ID               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Gallery Number          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Department              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ AccessionYear           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Object Name   

## Step 4: Filter and aggregate with SQL

This is where DuckDB really shines: instead of pulling everything into
`pandas` and then filtering, push the filtering and aggregation down into the
SQL query. DuckDB only reads what it needs to answer the question.


In [22]:
# How many objects does each department have?
duckdb.sql(f"""
    SELECT "Department", COUNT(*) AS n_objects
    FROM read_csv_auto('{DATA_PATH}')
    GROUP BY "Department"
    ORDER BY n_objects DESC
""").show(max_rows=25)


┌───────────────────────────────────────────┬───────────┐
│                Department                 │ n_objects │
│                  varchar                  │   int64   │
├───────────────────────────────────────────┼───────────┤
│ Drawings and Prints                       │    172630 │
│ European Sculpture and Decorative Arts    │     43051 │
│ Photographs                               │     37459 │
│ Asian Art                                 │     37000 │
│ Greek and Roman Art                       │     33726 │
│ Costume Institute                         │     31652 │
│ Egyptian Art                              │     27969 │
│ The American Wing                         │     18532 │
│ Islamic Art                               │     15573 │
│ Modern and Contemporary Art               │     14696 │
│ Arms and Armor                            │     13623 │
│ Arts of Africa, Oceania, and the Americas │     12367 │
│ Medieval Art                              │      7142 │
│ Ancient Near

In [23]:
# Find public-domain paintings from a specific culture/period, for example
duckdb.sql(f"""
    SELECT "Object ID", "Title", "Artist Display Name", "Object Date", "Department"
    FROM read_csv_auto('{DATA_PATH}')
    WHERE "Is Public Domain" = true
      AND "Classification" ILIKE '%painting%'
    ORDER BY "Object Date"
    LIMIT 20
""").show()


┌───────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────┐
│ Object ID │                                                   Title                                                    │      Artist Display Name       │                                     Object Date                                      │        Department        │
│   int64   │                                                  varchar                                                   │            varchar             │                                       varchar                                        │         varchar          │
├───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────┼───────────────────────────────────

## Step 5: Bring a small result into `pandas`

Once you've filtered down to a manageable subset — a few hundred or thousand
rows instead of half a million — it's completely safe to convert that result
into a `pandas` DataFrame for further analysis, plotting, exporting, etc.


In [25]:
result = duckdb.sql(f"""
    SELECT "Object ID", "Title", "Artist Display Name", "Object Date",
           "Department", "Classification", "Is Public Domain"
    FROM read_csv_auto('{DATA_PATH}')
    WHERE "Department" = 'European Paintings'
""")

df_paintings = result.df()   # <-- small, safe pandas DataFrame
print(type(df_paintings))
print(df_paintings.shape)
df_paintings.head()


<class 'pandas.DataFrame'>
(2626, 7)


,Object ID,Title,Artist Display Name,Object Date,Department,Classification,Is Public Domain
0,435570,A Ship in a Stormy Sea,Ivan Konstantinovich Aivazovsky (Hovhannes Aiv...,1892,European Paintings,Miniatures,True
1,435572,Saint Giles with Christ Triumphant over Satan ...,Miguel Alcañiz (or Miquel Alcanyís),ca. 1408,European Paintings,Paintings,True
2,435573,Flora and Zephyr,Jacopo Amigoni,1730s,European Paintings,Paintings,True
3,435574,The Crucifixion,Andrea di Bartolo,NaN,European Paintings,Paintings,True
4,435575,"Jérôme Bonaparte (1784–1860), King of Westphalia",Giacomo Andreoli,NaN,European Paintings,Miniatures,True


## Step 6 (bonus): The Polars alternative

[Polars](https://pola.rs/) is another modern option, with a `pandas`-like API
but built for performance and larger-than-memory data. The key is
`scan_csv()`, which builds a *lazy* query plan instead of reading the file
immediately — nothing actually loads until you call `.collect()`.


In [26]:
import polars as pl

# Build a lazy query -- nothing is read from disk yet
lazy_df = (
    pl.scan_csv(DATA_PATH)
    .filter(pl.col("Department") == "European Paintings")
    .select(["Object ID", "Title", "Artist Display Name", "Object Date"])
)

# Only now does Polars actually scan the file, and only the parts it needs
df_polars = lazy_df.collect()
print(df_polars.shape)
df_polars.head()


(2626, 4)


Object ID,Title,Artist Display Name,Object Date
i64,str,str,str
435570,"""A Ship in a Stormy Sea""","""Ivan Konstantinovich Aivazovsk…","""1892"""
435572,"""Saint Giles with Christ Triump…","""Miguel Alcañiz (or Miquel Alca…","""ca. 1408"""
435573,"""Flora and Zephyr""","""Jacopo Amigoni""","""1730s"""
435574,"""The Crucifixion""","""Andrea di Bartolo""",null
435575,"""Jérôme Bonaparte (1784–1860), …","""Giacomo Andreoli""",null


## Recap

| Tool | Loads full file into memory? | Good for |
|---|---|---|
| `pandas.read_csv()` | Yes (by default) | Small/medium files, or large files *with* `nrows=`/`chunksize=` |
| **DuckDB** | No — queries file directly | SQL-style filtering/aggregation on very large CSVs |
| **Polars** (`scan_csv`) | No — lazy evaluation | pandas-like syntax at larger-than-memory scale |

**Rule of thumb:** if a file is big enough that you're nervous about opening
it in Excel, reach for DuckDB or Polars *before* you reach for
`pandas.read_csv()` on the whole thing. Filter/aggregate first, then convert
the small result to `pandas` if you want to keep using familiar pandas syntax
for the rest of your analysis.
